# Aula 05 - Validação e Robustez do Modelo

Módulo 08 AdmTech, Computação · Semana 8 · Sprint 4 · Prof. José Romualdo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josercf/2026-2B-M08/blob/main/notebooks/aula05.ipynb)

## O que este notebook é

Laboratório executável da Aula 05, que alimenta o artefato A4 (Otimizador de Carteira com Fronteira Eficiente) e prepara a Ponderada 2. O otimizador da Aula 04 é rodado numa janela de treino e testado nos meses seguintes, fora da amostra. Roda no Colab pelo link acima ou localmente a partir do repositório clonado. Sem rede, a carga usa o snapshot versionado em `dados/`.

## 1. A cesta desde 2021

A cesta de referência tem 15 ativos da B3. O estudo começa em 04/01/2021, primeiro pregão de 2021: o GOLD11 só tem cotação a partir de 11/12/2020, e começar em 2021 deixa todos os 15 ativos presentes desde o primeiro dia e as janelas em anos civis inteiros. Dentro do período sobra uma única data com ausência, 24/10/2025, quando BOVA11, IVVB11 e GOLD11 ficaram sem cotação. Ela sai da cesta inteira (`dropna`), e o retorno do pregão seguinte cobre os dois dias.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

DADOS_LOCAL = os.path.join("..", "dados")
DADOS_BRUTOS = "https://raw.githubusercontent.com/josercf/2026-2B-M08/main/dados/"
PREGOES_ANO = 252


def ler_csv(nome, **kwargs):
    """Lê um CSV de dados/: do disco quando o repositório está clonado, da URL bruta no Colab.

    Com M08_OFFLINE=1 a queda para a rede fica desligada, como no CI.
    """
    local = os.path.join(DADOS_LOCAL, nome)
    if os.path.exists(local):
        return pd.read_csv(local, **kwargs)
    if os.environ.get("M08_OFFLINE") == "1":
        raise FileNotFoundError("%s não está em %s e M08_OFFLINE=1 desliga a rede" % (nome, DADOS_LOCAL))
    return pd.read_csv(DADOS_BRUTOS + nome, **kwargs)


precos = ler_csv("precos_ajustados.csv", index_col="data", parse_dates=["data"]).loc["2021-01-01":]
print("datas descartadas:", [d.date() for d in precos.index[precos.isna().any(axis=1)]])
precos = precos.dropna()
retornos = precos.pct_change().dropna()
cdi = ler_csv("cdi_diario.csv", index_col="data", parse_dates=["data"])["valor"] / 100
print("pregões de retorno:", len(retornos), "de", retornos.index[0].date(), "a", retornos.index[-1].date())

## 2. O otimizador da Aula 04

As mesmas três peças da Aula 04: estimar retorno esperado e covariância anualizados por 252, anualizar o CDI das datas da janela como taxa livre de risco e resolver com SLSQP, com pesos que somam 1 e sem venda a descoberto. A tolerância cai um degrau quando o SLSQP para com "Positive directional derivative for linesearch" já em cima do ótimo.

In [ ]:
def cdi_anual(datas):
    """CDI composto nas datas dadas e anualizado por 252 pregões."""
    taxas = cdi.reindex(datas)
    return (1 + taxas).prod() ** (PREGOES_ANO / len(taxas)) - 1


def estimar(janela):
    """Retorno esperado e covariância anuais a partir de retornos diários."""
    return janela.mean().to_numpy() * PREGOES_ANO, janela.cov().to_numpy() * PREGOES_ANO


def otimizar(mu, cov, rf, objetivo):
    """Pesos ótimos por SLSQP: soma 1, cada peso entre 0 e 1."""
    n = len(mu)
    if objetivo == "var_min":
        custo = lambda w: w @ cov @ w
    else:
        custo = lambda w: -(w @ mu - rf) / np.sqrt(w @ cov @ w)
    for tolerancia in (1e-12, 1e-10, 1e-8):
        resultado = minimize(custo, np.full(n, 1 / n), method="SLSQP", bounds=[(0, 1)] * n,
                             constraints=({"type": "eq", "fun": lambda w: w.sum() - 1},),
                             options={"ftol": tolerancia, "maxiter": 1000})
        if resultado.success:
            w = np.clip(resultado.x, 0, None)
            return w / w.sum()
    raise RuntimeError(resultado.message)


def esperado(w, mu, cov, rf):
    retorno, vol = w @ mu, np.sqrt(w @ cov @ w)
    return {"retorno": retorno, "vol": vol, "sharpe": (retorno - rf) / vol}


treino = retornos.loc["2023-01-01":"2024-12-31"]
mu, cov = estimar(treino)
rf_treino = cdi_anual(treino.index)
pesos = {objetivo: otimizar(mu, cov, rf_treino, objetivo) for objetivo in ("max_sharpe", "var_min")}
print("pregões de treino:", len(treino), "· CDI do treino: %.2f%%" % (100 * rf_treino))
(100 * pd.DataFrame(pesos, index=retornos.columns)).round(2).query("max_sharpe > 0.5 or var_min > 0.5")

## 3. Backtest fora da amostra

O teste usa o ano seguinte ao treino, 2025, que o otimizador nunca viu. Os pesos ficam congelados no teste: a carteira rende `retornos_teste @ w` a cada pregão. O patrimônio começa em 1 e acumula esses retornos, e o drawdown de cada dia é a queda do patrimônio desde o maior valor anterior. A divisão é no tempo: sortear pregões para o teste deixaria o treino enxergar dias posteriores a ele.

In [ ]:
def realizado(w, retornos_teste, cdi_teste):
    """Métricas fora da amostra da carteira de pesos constantes."""
    diario = pd.Series(retornos_teste.to_numpy() @ w, index=retornos_teste.index)
    retorno = diario.mean() * PREGOES_ANO
    vol = diario.std() * np.sqrt(PREGOES_ANO)
    rf = (1 + cdi_teste).prod() ** (PREGOES_ANO / len(cdi_teste)) - 1
    patrimonio = (1 + diario).cumprod()
    queda = patrimonio / patrimonio.cummax() - 1
    return {"retorno": retorno, "vol": vol, "sharpe": (retorno - rf) / vol,
            "drawdown_max": queda.min(), "data_fundo": queda.idxmin().date(),
            "acumulado": patrimonio.iloc[-1] - 1, "patrimonio": patrimonio, "drawdown": queda}


teste = retornos.loc["2025-01-01":"2025-12-31"]
cdi_teste = cdi.reindex(teste.index)
linhas = {}
backtest = {}
for objetivo, w in pesos.items():
    backtest[objetivo] = realizado(w, teste, cdi_teste)
    linhas[objetivo + ", esperado"] = esperado(w, mu, cov, rf_treino)
    linhas[objetivo + ", realizado"] = {k: backtest[objetivo][k] for k in ("retorno", "vol", "sharpe", "drawdown_max")}
cdi_acumulado = (1 + cdi.loc[teste.index[0]:teste.index[-1]]).prod() - 1
print("pregões de teste:", len(teste), "· CDI acumulado no teste: %.2f%%" % (100 * cdi_acumulado))
pd.DataFrame(linhas).T.round(4)

In [ ]:
fig, (topo, base) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for objetivo, r in backtest.items():
    r["patrimonio"].plot(ax=topo, label=objetivo)
    r["drawdown"].plot(ax=base, label=objetivo)
(1 + cdi.loc[teste.index[0]:teste.index[-1]]).cumprod().plot(ax=topo, label="CDI", linestyle="--")
topo.set_ylabel("R$ por R$ 1 aplicado")
base.set_ylabel("drawdown")
topo.legend()
plt.show()
for objetivo, r in backtest.items():
    print("%s: drawdown máximo %.2f%% em %s, acumulado %.2f%%"
          % (objetivo, 100 * r["drawdown_max"], r["data_fundo"], 100 * r["acumulado"]))

### Pergunta para a dupla

1. Qual das quatro métricas esperadas ficou mais perto do realizado, e qual ficou mais longe?
2. O CDI do teste foi diferente do CDI do treino. Em que direção isso empurra o Sharpe realizado?
3. Que número do A4 do grupo ainda não foi testado fora da amostra?

<details>
<summary>Resposta</summary>

1. A volatilidade ficou mais perto nas duas carteiras. O retorno esperado do máximo Sharpe foi o que mais errou: o otimizador concentra peso nos ativos que mais renderam no treino, e boa parte daquele retorno era acaso da janela.
2. O CDI subiu, e CDI maior reduz o Sharpe, porque ele é subtraído do retorno antes da divisão pela volatilidade.
3. Em geral, o retorno e o Sharpe das duas carteiras ótimas e a própria fronteira: todos foram calculados e avaliados na mesma janela.

</details>

## 4. A janela de treino desliza

Um único ano de teste pode ser sorte ou azar. As janelas deslizantes repetem o mesmo experimento várias vezes: treino de 24 meses, teste nos 12 meses seguintes, e a janela anda um mês de cada vez, do treino iniciado em 01/2021 até o último teste que cabe no snapshot.

In [ ]:
def janelas_deslizantes(meses_treino=24, meses_teste=12):
    ultimo = retornos.index[-1]
    inicio = pd.Timestamp("2021-01-01")
    janelas = []
    while True:
        fim_treino = inicio + pd.DateOffset(months=meses_treino) - pd.Timedelta(days=1)
        inicio_teste = fim_treino + pd.Timedelta(days=1)
        fim_teste = inicio_teste + pd.DateOffset(months=meses_teste) - pd.Timedelta(days=1)
        if fim_teste > ultimo + pd.offsets.MonthEnd(0):
            return janelas
        janelas.append(((inicio, fim_treino), (inicio_teste, fim_teste)))
        inicio = inicio + pd.DateOffset(months=1)


registros = []
for (ti, tf), (si, sf) in janelas_deslizantes():
    r_treino, r_teste = retornos.loc[ti:tf], retornos.loc[si:sf]
    mu_j, cov_j = estimar(r_treino)
    rf_j = cdi_anual(r_treino.index)
    for objetivo in ("max_sharpe", "var_min"):
        w = otimizar(mu_j, cov_j, rf_j, objetivo)
        registros.append({"treino": ti.strftime("%m/%Y"), "objetivo": objetivo, "pesos": w,
                          "prometido": esperado(w, mu_j, cov_j, rf_j)["sharpe"],
                          "entregue": realizado(w, r_teste, cdi.reindex(r_teste.index))["sharpe"]})
deslizantes = pd.DataFrame(registros)
print("janelas:", deslizantes["treino"].nunique())
deslizantes.assign(abaixo=deslizantes["entregue"] < deslizantes["prometido"]).groupby("objetivo").agg(
    abaixo_do_prometido=("abaixo", "sum"), prometido_mediano=("prometido", "median"),
    entregue_mediano=("entregue", "median")).round(2)

## 5. Estabilidade dos pesos

A carteira trocada entre duas versões de pesos é metade da soma, ativo a ativo, da diferença absoluta entre eles: zero é a mesma carteira, 100% é trocar tudo. É a fração do patrimônio que precisaria mudar de mãos para sair de uma versão e chegar na outra.

In [ ]:
def carteira_trocada(w1, w2):
    return 0.5 * np.abs(np.asarray(w1) - np.asarray(w2)).sum()


for objetivo in ("max_sharpe", "var_min"):
    w = deslizantes.loc[deslizantes["objetivo"] == objetivo, "pesos"].tolist()
    mensal = [carteira_trocada(a, b) for a, b in zip(w[:-1], w[1:])]
    print("%s: carteira trocada de um mês para o seguinte, mediana %.2f%%, máximo %.2f%%"
          % (objetivo, 100 * np.median(mensal), 100 * np.max(mensal)))


def pesos_na_janela(inicio, fim, objetivo):
    janela = retornos.loc[inicio:fim]
    mu_j, cov_j = estimar(janela)
    return otimizar(mu_j, cov_j, cdi_anual(janela.index), objetivo)


for objetivo in ("max_sharpe", "var_min"):
    a = pesos_na_janela("2021-01-01", "2022-12-31", objetivo)
    b = pesos_na_janela("2023-01-01", "2024-12-31", objetivo)
    print("%s: 2021-2022 contra 2023-2024, carteira trocada %.2f%%" % (objetivo, 100 * carteira_trocada(a, b)))

## 6. Sensibilidade: retorno perturbado contra Ledoit-Wolf

Duas mudanças na mesma janela de treino. A primeira soma ou tira 5 pontos percentuais de cada retorno esperado, com o sinal sorteado por semente fixa. A segunda troca a covariância amostral pela de Ledoit e Wolf (2004), que encolhe a matriz na direção de uma matriz diagonal; a função abaixo faz a mesma conta do `sklearn.covariance.LedoitWolf`, escrita em numpy para não exigir o scikit-learn. Antes de rodar, registre qual das duas você espera que mexa mais nos pesos.

In [ ]:
def ledoit_wolf(x):
    """Covariância diária encolhida de Ledoit e Wolf, alvo mu vezes identidade."""
    x = np.asarray(x, dtype=float)
    x = x - x.mean(axis=0)
    n, p = x.shape
    emp = x.T @ x / n
    alvo = np.trace(emp) / p
    delta = ((emp - alvo * np.eye(p)) ** 2).sum() / p
    beta = min((((x ** 2).T @ (x ** 2)) / n - emp ** 2).sum() / (p * n), delta)
    encolhimento = beta / delta
    return encolhimento * alvo * np.eye(p) + (1 - encolhimento) * emp, encolhimento


cov_lw, encolhimento = ledoit_wolf(treino.to_numpy())
cov_lw = cov_lw * PREGOES_ANO
sinais = np.random.default_rng(8).choice([-1.0, 1.0], len(mu))
mu_perturbado = mu + 0.05 * sinais
erro_padrao = np.sqrt(np.diag(cov)) / np.sqrt(len(treino) / PREGOES_ANO)
print("encolhimento de Ledoit-Wolf: %.2f%%" % (100 * encolhimento))
print("erro padrão mediano do retorno esperado anual: %.2f%%" % (100 * np.median(erro_padrao)))
for objetivo in ("max_sharpe", "var_min"):
    base = pesos[objetivo]
    perturbado = otimizar(mu_perturbado, cov, rf_treino, objetivo)
    encolhido = otimizar(mu, cov_lw, rf_treino, objetivo)
    print("%s: carteira trocada com retorno perturbado %.2f%%, com Ledoit-Wolf %.2f%%"
          % (objetivo, 100 * carteira_trocada(base, perturbado), 100 * carteira_trocada(base, encolhido)))

### Registre a decisão

Cada aluno grava, a partir do [modelo de ADR do curso](https://gist.github.com/josercf/fc714afe20773401dc9d56f65174b441), o ADR que declara quanto confiar no otimizador do grupo. O ADR cita pelo menos um número fora da amostra medido nos dados do grupo (Sharpe prometido contra entregue, drawdown ou carteira trocada entre janelas) e registra a janela de dados e o estimador de covariância escolhidos, cada um com o impacto numérico.

## 7. Desafio para o A4

a. Reotimizar a carteira de máximo Sharpe do grupo em três janelas de 12 meses seguidas.
b. Medir quanto cada peso varia entre as três janelas, por exemplo pela diferença entre o maior e o menor peso de cada ativo, e a carteira trocada entre janelas vizinhas.
c. Escrever, em um parágrafo, o que essa variação significa para a recomendação ao comitê do parceiro.

A célula abaixo já otimiza as três janelas da cesta de referência. Troque a cesta pela do grupo e complete a medida de variação.

In [ ]:
TRES_JANELAS = [("2022-01-01", "2022-12-31"), ("2023-01-01", "2023-12-31"), ("2024-01-01", "2024-12-31")]
tabela = pd.DataFrame({inicio[:4]: pesos_na_janela(inicio, fim, "max_sharpe") for inicio, fim in TRES_JANELAS},
                      index=retornos.columns)
print((100 * tabela[(tabela > 0.005).any(axis=1)]).round(2))

# Complete: a variação de cada peso entre as três janelas e a carteira trocada entre janelas vizinhas.
variacao_por_ativo = None